### Single Scene

In [9]:
import os
import ast
import json
import random
from scene_annot import extract_annotation

DATA_DIR = '../scene'
OUTPUT_DIR = '../../data/qwen2.5-vl'

TEST_RATIO = 0.1

ANNOT_DIRS = {
    'free_fall': ['free_fall/output'],
    'slope': [],
    'ripple': [],
    'obj_moving': ['obj_moving/output'],
    'obj_interaction': []
}

for scene_type in ANNOT_DIRS.keys():
    descs = []
    # qas = []
    
    desc_cnt = 0
    qa_cnt = 0
    
    for subfolder in ANNOT_DIRS[scene_type]:
        sub_path = os.path.join(DATA_DIR, subfolder)
        i = 0
        while True:
            log_path = os.path.join(sub_path, str(i), "params.log")
            if not os.path.isfile(log_path):
                break

            with open(log_path, "r", encoding="utf-8") as f:
                content = f.read().strip()
                try:
                    params_dict = ast.literal_eval(content)
                except (ValueError, SyntaxError):
                    try:
                        params_dict = json.loads(content)
                    except json.JSONDecodeError:
                        print(f"Warning: cannot parse '{log_path}'.")
                        break

            # descs
            annotation = extract_annotation(params_dict, scene_type)

            desc = {
                "video": f"{subfolder}/{i}/render.mkv", #비디오 제목에 따라 지정
                "conversations": [
                    {
                        "from": "human",
                        "value": "<video>\nDescribe the video."
                    },
                    {
                        "from": "gpt",
                        "value": annotation
                    }
                ]
            }

            descs.append(desc)
            desc_cnt += 1

            # qas
            # qas.append({
            #     # TODO
            # })
            # qa_cnt += 1

            i += 1

    DESC_TRAIN_SIZE = int(desc_cnt * (1 - TEST_RATIO))
    # QA_TRAIN_SIZE = int(qa_cnt * (1 - TEST_RATIO))

    desc_test_path = os.path.join(OUTPUT_DIR, scene_type, "desc_test.json")
    desc_train_path = os.path.join(OUTPUT_DIR, scene_type, "desc_train.json")

    # qa_test_path = os.path.join(OUTPUT_DIR, "qa_test.json")
    # qa_train_path = os.path.join(OUTPUT_DIR, "qa_train.json")

    if not os.path.exists(OUTPUT_DIR):
        os.makedirs(OUTPUT_DIR)
    if not os.path.exists(os.path.join(OUTPUT_DIR, scene_type)):
        os.makedirs(os.path.join(OUTPUT_DIR, scene_type))

    descs = random.sample(descs, len(descs))  # Shuffle the descriptions
    # qas = random.sample(qas, len(qas))  # Shuffle the QAs

    with open(desc_train_path, "w", encoding="utf-8") as f:
        json.dump(descs[:DESC_TRAIN_SIZE], f, indent=4, ensure_ascii=False)
    with open(desc_test_path, "w", encoding="utf-8") as f:
        json.dump(descs[DESC_TRAIN_SIZE:], f, indent=4, ensure_ascii=False)

    # with open(qa_train_path, "w", encoding="utf-8") as f:
    #     json.dump(qas[:QA_TRAIN_SIZE], f, indent=4, ensure_ascii=False)
    # with open(qa_test_path, "w", encoding="utf-8") as f:
    #     json.dump(qas[QA_TRAIN_SIZE:], f, indent=4, ensure_ascii=False)

    print(f"Processed {scene_type}: {desc_cnt} descriptions, {qa_cnt} QAs.")



Processed free_fall: 480 descriptions, 0 QAs.
Processed slope: 0 descriptions, 0 QAs.
Processed ripple: 0 descriptions, 0 QAs.
Processed obj_moving: 5 descriptions, 0 QAs.
Processed obj_interaction: 0 descriptions, 0 QAs.


### Multi Scene

In [11]:
import os
import ast
import json
import random
from scene_annot import extract_annotation
from utils import concat

DATA_DIR = '../scene'
OUTPUT_DIR = '../../data/qwen2.5-vl'

TEST_RATIO = 0.1

ANNOT_DIRS = {
    'free_fall': ['free_fall/output'],
    'slope': [],
    'ripple': [],
    'obj_moving': ['obj_moving/output'],
    'obj_interaction': []
}

params = []

for scene_type in ANNOT_DIRS.keys():
    for subfolder in ANNOT_DIRS[scene_type]:
        sub_path = os.path.join(DATA_DIR, subfolder)
        i = 0
        while True:
            log_path = os.path.join(sub_path, str(i), "params.log")
            if not os.path.isfile(log_path):
                break

            with open(log_path, "r", encoding="utf-8") as f:
                content = f.read().strip()
                try:
                    params_dict = ast.literal_eval(content)
                except (ValueError, SyntaxError):
                    try:
                        params_dict = json.loads(content)
                    except json.JSONDecodeError:
                        print(f"Warning: cannot parse '{log_path}'.")
                        break

            params.append({
                "scene_type": scene_type,
                "video": f"{subfolder}/{i}/render.mkv",
                "params": params_dict
            })
            i += 1

print(f"Total scenes: {len(params)}")

TRAIN_SIZE = int(len(params) * (1 - TEST_RATIO))

params = random.sample(params, len(params))  # Shuffle the parameters
train_params = params[:TRAIN_SIZE]
test_params = params[TRAIN_SIZE:]

TRAIN_SAMPLE_SIZE = 1
TEST_SAMPLE_SIZE = 1

train_descs, test_descs = [], []
train_qas, test_qas = [], []
descs, qas = [], []

os.makedirs(f"{DATA_DIR}/concat", exist_ok=True)
os.makedirs(f"{DATA_DIR}/concat/train", exist_ok=True)
os.makedirs(f"{DATA_DIR}/concat/test", exist_ok=True)

for i in range(TRAIN_SAMPLE_SIZE):
    choices = random.sample(train_params, 2)
    os.makedirs(f"{DATA_DIR}/concat/train/{i}", exist_ok=True)

    concat(f"{DATA_DIR}/{choices[0]['video']}", f"{DATA_DIR}/{choices[1]['video']}", f"{DATA_DIR}/concat/train/{i}/row.mkv", f"{DATA_DIR}/concat/train/{i}/col.mkv")

    annot1 = extract_annotation(choices[0]['params'], choices[0]['scene_type'])
    annot2 = extract_annotation(choices[1]['params'], choices[1]['scene_type'])

    for e in ['row', 'col']:
        train_descs.append({
            "video": f"concat/{i}/{e}.mkv",
            "conversations": [
                {
                    "from": "human",
                    "value": f"<video>\nDescribe the video."
                },
                {
                    "from": "gpt",
                    "value": f"First Scene: {annot1}\nSecond Scene: {annot2}"
                }
            ]
        })
    
    visco1 = choices[0]['params']['viscosity']
    visco2 = choices[1]['params']['viscosity']
    visco1 = 0 if visco1 < 0.001 else 1 if visco1 < 0.01 else 2
    visco2 = 0 if visco2 < 0.001 else 1 if visco2 < 0.01 else 2

    if visco1 == visco2:
        continue
    else:
        for e in ['row', 'col']:
            train_qas.append({
                "video": f"concat/{i}/{e}.mkv",
                "conversations": [
                    {
                        "from": "human",
                        "value": f"<video>\nWhich scene has higher viscosity? First or Second?"
                    },
                    {
                        "from": "gpt",
                        "value": "The first scene." if visco1 > visco2 else "The second scene."
                    }
                ]
            })

for i in range(TEST_SAMPLE_SIZE):
    choices = random.sample(test_params, 2)
    os.makedirs(f"{DATA_DIR}/concat/test/{i}", exist_ok=True)
    
    concat(f"{DATA_DIR}/{choices[0]['video']}", f"{DATA_DIR}/{choices[1]['video']}", f"{DATA_DIR}/concat/test/{i}/row.mkv", f"{DATA_DIR}/concat/test/{i}/col.mkv")

    annot1 = extract_annotation(choices[0]['params'], choices[0]['scene_type'])
    annot2 = extract_annotation(choices[1]['params'], choices[1]['scene_type'])

    for e in ['row', 'col']:
        test_descs.append({
            "video": f"concat/{i}/{e}.mkv",
            "conversations": [
                {
                    "from": "human",
                    "value": f"<video>\nDescribe the video."
                },
                {
                    "from": "gpt",
                    "value": f"First Scene: {annot1}\nSecond Scene: {annot2}"
                }
            ]
        })
    
    visco1 = choices[0]['params']['viscosity']
    visco2 = choices[1]['params']['viscosity']
    visco1 = 0 if visco1 < 0.001 else 1 if visco1 < 0.01 else 2
    visco2 = 0 if visco2 < 0.001 else 1 if visco2 < 0.01 else 2

    if visco1 == visco2:
        continue
    else:
        for e in ['row', 'col']:
            test_qas.append({
                "video": f"concat/{i}/{e}.mkv",
                "conversations": [
                    {
                        "from": "human",
                        "value": f"<video>\nWhich scene has higher viscosity? First or Second?"
                    },
                    {
                        "from": "gpt",
                        "value": "The first scene." if visco1 > visco2 else "The second scene."
                    }
                ]
            })


# Save the results
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)
if not os.path.exists(os.path.join(OUTPUT_DIR, "concat")):
    os.makedirs(os.path.join(OUTPUT_DIR, "concat"))

with open(os.path.join(OUTPUT_DIR, "concat/desc_train.json"), "w", encoding="utf-8") as f:
    json.dump(train_descs, f, indent=4, ensure_ascii=False)
with open(os.path.join(OUTPUT_DIR, "concat/desc_test.json"), "w", encoding="utf-8") as f:
    json.dump(test_descs, f, indent=4, ensure_ascii=False)
with open(os.path.join(OUTPUT_DIR, "concat/qa_train.json"), "w", encoding="utf-8") as f:
    json.dump(train_qas, f, indent=4, ensure_ascii=False)
with open(os.path.join(OUTPUT_DIR, "concat/qa_test.json"), "w", encoding="utf-8") as f:
    json.dump(test_qas, f, indent=4, ensure_ascii=False)

print(f"Processed concatenated scenes: {len(train_descs)} train descriptions, {len(test_descs)} test descriptions.")
print(f"Processed concatenated scenes: {len(train_qas)} train QAs, {len(test_qas)} test QAs.")


Total scenes: 1056



                                                                  

                                                               


                                                            



                                                                      





frame_index:   4%|▍         | 2/50 [54:05<00:04, 11.48it/s, now=None]
                                                                  

                                                               


                                                            



                                                                      





frame_index:   4%|▍         | 2/50 [54:05<00:04, 11.48it/s, now=None]

MoviePy - Building video ../scene/concat/train/0/row.mkv.
MoviePy - Writing video ../scene/concat/train/0/row.mkv












































































































































































































                                                                  

                                                               


                                                            



                                                                      





frame_index:   4%|▍         | 2/50 [54:13<00:04, 11.48it/s, now=None]
                                                                  

                                                               


                                                            



                                                                      





frame_index:   4%|▍         | 2/50 [54:14<00:04, 11.48it/s, now=None]
                                                                  

                                           

MoviePy - Done !
MoviePy - video ready ../scene/concat/train/0/row.mkv
MoviePy - Building video ../scene/concat/train/0/col.mkv.
MoviePy - Writing video ../scene/concat/train/0/col.mkv



...

MoviePy - Building video ../scene/concat/test/0/row.mkv.
MoviePy - Writing video ../scene/concat/test/0/row.mkv












































































































































































































                                                                  

                                                               


                                                            



                                                                      





frame_index:   4%|▍         | 2/50 [54:31<00:04, 11.48it/s, now=None]
                                                                  

                                                               


                                                            



                                                                      





frame_index:   4%|▍         | 2/50 [54:31<00:04, 11.48it/s, now=None]
                                                                  

                                           

MoviePy - Done !
MoviePy - video ready ../scene/concat/test/0/row.mkv
MoviePy - Building video ../scene/concat/test/0/col.mkv.
MoviePy - Writing video ../scene/concat/test/0/col.mkv
















































































































































































































                                                                  

                                                               


                                                            



                                                                      





frame_index:   4%|▍         | 2/50 [54:39<00:04, 11.48it/s, now=None]
                                                                  

                                                               


                                                            



                                                                      





frame_index:   4%|▍         | 2/50 [54:39<00:04, 11.48it/s, now=None]

MoviePy - Done !
MoviePy - video ready ../scene/concat/test/0/col.mkv
Processed concatenated scenes: 2 train descriptions, 2 test descriptions.
Processed concatenated scenes: 0 train QAs, 2 test QAs.
